# Servidor Unificado de Inferencia (M1 + M2 + M3) en RunPod
**IAM-811 Machine Learning y Aprendizaje Profundo** · Laboratorio 3: Sistema de Control de Peaje

Basado en la arquitectura del notebook de clase `week_2_servidor_pose_runpod.ipynb`. Levanta la API FastAPI que sirve los **3 modelos en un único endpoint (`/predict`)**:
- **M1:** Detección de placas + EasyOCR + Normalización
- **M2:** Segmentación de instancias de vehículos (`YOLO11-seg`: `moto`, `auto`, `minibus`, `camion`, `bus`)
- **M3:** Keypoints sobre las ruedas (`YOLO11-pose`) para estimar el número de ejes

**Requisito en RunPod:** Exponer el puerto HTTP **8000** en la configuración del Pod.

In [ ]:
# 1. Configuración y URL pública del proxy de RunPod
import os, secrets

PUERTO = 8000
API_TOKEN = os.environ.get("API_TOKEN", "token_secreto_inferencia_peaje_2026")
os.environ["API_TOKEN"] = API_TOKEN

POD_ID = os.environ.get("RUNPOD_POD_ID")
if POD_ID:
    URL_PUBLICA = f"https://{POD_ID}-{PUERTO}.proxy.runpod.net"
else:
    URL_PUBLICA = f"http://localhost:{PUERTO}"

print("INFERENCE_URL (para configurar en el backend web):", URL_PUBLICA)
print("INFERENCE_API_TOKEN:", API_TOKEN)

In [ ]:
# 2. Instalación de dependencias
!pip install -q ultralytics==8.3.27 easyocr==1.7.2 fastapi==0.115.4 "uvicorn[standard]==0.32.0" python-multipart opencv-python-headless

In [ ]:
# 3. Ejecución del script de diagnóstico (Bonificación +5 pts Gestión de Entornos)
from diagnosticar_entorno import diagnosticar
diagnosticar()

In [ ]:
# 4. Carga e inicio del servidor FastAPI en hilo de fondo (escucha en 0.0.0.0:8000)
import threading, time, uvicorn
from app_inferencia import app, obtener_motor

motor = obtener_motor()
print("Modelos cargados:", motor.m1_nombre, "|", motor.m2_nombre, "|", motor.m3_nombre)

if "servidor" in globals() and servidor.started and not servidor.should_exit:
    print("El servidor ya está corriendo.")
else:
    config = uvicorn.Config(app, host="0.0.0.0", port=PUERTO, log_level="info")
    servidor = uvicorn.Server(config)
    hilo_servidor = threading.Thread(target=servidor.run, daemon=True)
    hilo_servidor.start()
    for _ in range(50):
        if servidor.started:
            break
        time.sleep(0.1)
    print("Servidor iniciado:", servidor.started)
    print("Endpoint público de inferencia:", URL_PUBLICA)

In [ ]:
# 5. Prueba en vivo del endpoint /predict (para mostrar en el video demostración - Sección 8 y 10)
import cv2, numpy as np, urllib.request, json

# Generar imagen de prueba de vehículo con placa boliviana 1852PHD
img_demo = np.full((480, 640, 3), 220, dtype=np.uint8)
cv2.rectangle(img_demo, (90, 140), (550, 360), (60, 90, 170), -1)
cv2.circle(img_demo, (180, 365), 38, (30, 30, 30), -1)
cv2.circle(img_demo, (460, 365), 38, (30, 30, 30), -1)
cv2.rectangle(img_demo, (235, 275), (405, 330), (250, 250, 250), -1)
cv2.putText(img_demo, "1852 PHD", (245, 315), cv2.FONT_HERSHEY_SIMPLEX, 0.95, (15, 15, 15), 3)

_, jpg_bytes = cv2.imencode(".jpg", img_demo)
req = urllib.request.Request(
    f"http://localhost:{PUERTO}/predict",
    data=jpg_bytes.tobytes(),
    headers={"Content-Type": "image/jpeg", "X-API-Token": API_TOKEN}
)
resp = json.loads(urllib.request.urlopen(req, timeout=45).read())
resp_sin_b64 = {k: v for k, v in resp.items() if k != "imagen_anotada_b64"}
print(json.dumps(resp_sin_b64, indent=2, ensure_ascii=False))